# Adversarial Data Protection Gradio Demo

Upload one or more images, choose a protection technique, and inspect the protected output, amplified perturbation map, image-quality metrics, and technique-specific model effect metrics. Resize mode is the default for Colab T4; patch mode is available as a slower advanced option.

In [ ]:
# Cell 1 - Setup (KHONG thay doi thu tu)
import subprocess
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])

import gc
import os

import torch
import torch.nn.functional as F
import torchvision
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import gradio as gr

from src.evaluation import compute_linf, compute_psnr, compute_ssim
from src.models import get_surrogate_resnet50
from src.pipeline import (
    image_to_tensor,
    image_to_tensor_no_resize,
    protect_patches,
    tensor_to_image,
)
from src.techniques.cloaking import cloak_images
from src.techniques.nightshade import get_text_embedding, load_clip_model, poison_images

TECH_UNLEARNABLE = "Unlearnable Demo"
TECH_CLOAKING = "General Feature-space Cloaking"
TECH_CONCEPT = "Concept Poisoning (Nightshade-style)"

surrogate = get_surrogate_resnet50(device)
clip_model = None

NOISE_DICT_PATH = "results/unlearnable_noise_dict.pt"
unlearnable_noise_dict = None
if os.path.exists(NOISE_DICT_PATH):
    unlearnable_noise_dict = torch.load(NOISE_DICT_PATH, map_location=device)
    print(f"[OK] Loaded {NOISE_DICT_PATH}")
else:
    print(f"[WARN] Missing {NOISE_DICT_PATH}. Run notebook_experiment.ipynb first for Unlearnable mode.")


def _get_clip_model():
    global clip_model
    if clip_model is None:
        clip_model, _ = load_clip_model("ViT-B/32", device)
    return clip_model


def _load_images(files):
    if not files:
        raise gr.Error("Please upload at least one image.")
    images = []
    for file_obj in files:
        file_path = getattr(file_obj, "name", file_obj)
        images.append(Image.open(file_path).convert("RGB"))
    return images


def resize_cifar_noise_for_ui(noise, target_hw, epsilon, source_epsilon=0.03):
    noise = noise.unsqueeze(0).float().to(device)
    noise = F.interpolate(noise, size=target_hw, mode="bilinear", align_corners=False)
    return noise * (float(epsilon) / source_epsilon)


def _protect_unlearnable_demo(batch, epsilon):
    if unlearnable_noise_dict is None:
        raise gr.Error(
            f"Missing {NOISE_DICT_PATH}. Run notebook_experiment.ipynb first to generate precomputed CIFAR noise."
        )
    noise = resize_cifar_noise_for_ui(
        unlearnable_noise_dict[0],
        target_hw=batch.shape[-2:],
        epsilon=epsilon,
    ).to(batch.device)
    return torch.clamp(batch + noise, 0.0, 1.0)


def _protect_cloaking(batch, epsilon, pgd_steps):
    return cloak_images(
        surrogate,
        batch,
        epsilon=epsilon,
        pgd_steps=pgd_steps,
        pgd_alpha=max(epsilon / 10, 1 / 255),
        target_mode="max_dist",
        device=device,
    )


def _protect_concept_poisoning(batch, epsilon, pgd_steps, target_concept):
    model = _get_clip_model()
    return poison_images(
        model,
        batch,
        target_concept=target_concept,
        epsilon=epsilon,
        pgd_steps=pgd_steps,
        pgd_alpha=max(epsilon * 2 / pgd_steps, 1 / 255),
        device=device,
    )


def _noise_image(x_orig, x_protected, scale, original_size=None):
    noise_vis = ((x_protected.detach().cpu() - x_orig.detach().cpu()) * scale + 0.5).clamp(0, 1)
    return tensor_to_image(noise_vis, original_size=original_size)


def _metric_tensor(x, size=224):
    if x.shape[-2:] == (size, size):
        return x
    return F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)


def _cloaking_effect_metrics(x_orig, x_protected):
    x_orig_m = _metric_tensor(x_orig).to(device)
    x_prot_m = _metric_tensor(x_protected).to(device)
    surrogate.eval()
    with torch.no_grad():
        f_orig = F.normalize(surrogate(x_orig_m).float(), dim=1)
        f_prot = F.normalize(surrogate(x_prot_m).float(), dim=1)
        cosine = F.cosine_similarity(f_orig, f_prot, dim=1)
        l2_shift = torch.norm(f_prot - f_orig, p=2, dim=1)
    return {
        "feature_cosine_orig_protected": cosine.detach().cpu(),
        "feature_l2_shift": l2_shift.detach().cpu(),
    }


def _clip_similarity_metrics(x_orig, x_protected, target_concept):
    model = _get_clip_model()
    x_orig_m = _metric_tensor(x_orig).to(device)
    x_prot_m = _metric_tensor(x_protected).to(device)
    target_emb = get_text_embedding(model, target_concept, device)
    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
    clip_std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)
    with torch.no_grad():
        orig_emb = F.normalize(model.encode_image((x_orig_m - clip_mean) / clip_std).float(), dim=-1)
        prot_emb = F.normalize(model.encode_image((x_prot_m - clip_mean) / clip_std).float(), dim=-1)
        before = F.cosine_similarity(orig_emb, target_emb.expand_as(orig_emb), dim=1)
        after = F.cosine_similarity(prot_emb, target_emb.expand_as(prot_emb), dim=1)
    return {
        "clip_target_similarity_before": before.detach().cpu(),
        "clip_target_similarity_after": after.detach().cpu(),
        "clip_target_similarity_delta": (after - before).detach().cpu(),
    }


def _quality_lines(x_orig, x_protected):
    return [
        f"PSNR(avg): {compute_psnr(x_orig, x_protected):.2f} dB",
        f"SSIM(avg): {compute_ssim(x_orig, x_protected):.4f}",
        f"Linf(max): {compute_linf(x_orig, x_protected):.4f}",
    ]


def _effect_lines(technique, x_orig, x_protected, target_concept):
    lines = []
    if technique == TECH_CLOAKING:
        metrics = _cloaking_effect_metrics(x_orig, x_protected)
        cos = metrics["feature_cosine_orig_protected"]
        shift = metrics["feature_l2_shift"]
        lines.extend([
            f"Feature cosine orig/protected avg: {cos.mean().item():.4f}",
            f"Feature L2 shift avg: {shift.mean().item():.4f}",
            "Multi-image resize mode: max_dist target is selected across uploaded images; single-image fallback uses away mode.",
        ])
    elif technique == TECH_CONCEPT:
        metrics = _clip_similarity_metrics(x_orig, x_protected, target_concept)
        before = metrics["clip_target_similarity_before"]
        after = metrics["clip_target_similarity_after"]
        delta = metrics["clip_target_similarity_delta"]
        lines.extend([
            f"Target concept: {target_concept}",
            f"CLIP target similarity before avg: {before.mean().item():.4f}",
            f"CLIP target similarity after avg: {after.mean().item():.4f}",
            f"CLIP target similarity delta avg: {delta.mean().item():+.4f}",
            "This is a CLIP-space proxy, not full Stable Diffusion/Nightshade training evaluation.",
        ])
    else:
        lines.extend([
            "Unlearnable UI: demo-only using resized CIFAR-10 class-wise noise class 0.",
            "Real Unlearnable evaluation is in notebook_experiment.ipynb: train victim on protected train set, test on clean test set.",
        ])
    return lines


def _protect_resize_batch(images, technique, epsilon, target_concept, pgd_steps, noise_scale):
    original_sizes = [img.size for img in images]
    x_orig = torch.cat([image_to_tensor(img) for img in images], dim=0).to(device)

    if technique == TECH_UNLEARNABLE:
        x_protected = _protect_unlearnable_demo(x_orig, epsilon)
    elif technique == TECH_CLOAKING:
        x_protected = _protect_cloaking(x_orig, epsilon, pgd_steps)
    elif technique == TECH_CONCEPT:
        x_protected = _protect_concept_poisoning(x_orig, epsilon, pgd_steps, target_concept)
    else:
        raise gr.Error(f"Unknown technique: {technique}")

    originals, protected, noises = [], [], []
    for idx, img in enumerate(images):
        originals.append((img, f"Original {idx + 1}"))
        protected.append((tensor_to_image(x_protected[idx], original_size=original_sizes[idx]), f"Protected {idx + 1}"))
        noises.append((_noise_image(x_orig[idx], x_protected[idx], noise_scale, original_size=original_sizes[idx]), f"Noise x{noise_scale:g} {idx + 1}"))
    return originals, protected, noises, x_orig.detach().cpu(), x_protected.detach().cpu()


def _protect_patch_images(images, technique, epsilon, target_concept, pgd_steps, noise_scale):
    originals, protected, noises = [], [], []
    x_orig_all, x_prot_all = [], []

    if technique == TECH_UNLEARNABLE:
        technique_fn = lambda batch: _protect_unlearnable_demo(batch, epsilon)
    elif technique == TECH_CLOAKING:
        technique_fn = lambda batch: _protect_cloaking(batch, epsilon, pgd_steps)
    elif technique == TECH_CONCEPT:
        technique_fn = lambda batch: _protect_concept_poisoning(batch, epsilon, pgd_steps, target_concept)
    else:
        raise gr.Error(f"Unknown technique: {technique}")

    for idx, img in enumerate(images):
        out_img, x_protected = protect_patches(
            img,
            technique_fn,
            batch_size=4,
            patch_size=224,
            device=device,
        )
        x_orig = image_to_tensor_no_resize(img).clamp(0, 1)
        originals.append((img, f"Original {idx + 1}"))
        protected.append((out_img, f"Protected {idx + 1}"))
        noises.append((_noise_image(x_orig.squeeze(0), x_protected.squeeze(0), noise_scale), f"Noise x{noise_scale:g} {idx + 1}"))
        x_orig_all.append(_metric_tensor(x_orig))
        x_prot_all.append(_metric_tensor(x_protected))

    return originals, protected, noises, torch.cat(x_orig_all, dim=0), torch.cat(x_prot_all, dim=0)


def protect_images(files, technique, epsilon, target_concept, pgd_steps, processing_mode, noise_scale):
    images = _load_images(files)
    epsilon = float(epsilon)
    pgd_steps = int(pgd_steps)
    noise_scale = float(noise_scale)
    concept = target_concept.strip() if target_concept else "a photo of a cat"
    processing_mode = processing_mode.lower()

    if processing_mode == "patch":
        originals, protected, noises, x_orig, x_protected = _protect_patch_images(
            images, technique, epsilon, concept, pgd_steps, noise_scale
        )
        mode_note = "Patch mode: each image is processed independently in 224x224 patch batches; this is slower but preserves more detail."
    else:
        originals, protected, noises, x_orig, x_protected = _protect_resize_batch(
            images, technique, epsilon, concept, pgd_steps, noise_scale
        )
        mode_note = "Resize mode: all uploaded images are batched at 224x224; General Cloaking can select max_dist targets across the uploaded batch."

    lines = [
        f"Technique: {technique}",
        f"Images: {len(images)}",
        f"Epsilon: {epsilon:.4f}",
        f"PGD steps: {pgd_steps}",
        mode_note,
        "",
        *(_quality_lines(x_orig, x_protected)),
        "",
        *(_effect_lines(technique, x_orig, x_protected, concept)),
    ]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return originals, protected, noises, "\n".join(lines)


with gr.Blocks(title="Image Protection Demo") as demo:
    gr.Markdown("# Image Protection Demo")
    gr.Markdown(
        "Upload one or more images. Resize mode is fastest; patch mode keeps more detail but can be slow. "
        "Unlearnable is a demo-only CIFAR noise visualization. Concept Poisoning is a CLIP-space Nightshade-style proxy."
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_files = gr.Files(
                label="Upload image(s)",
                file_count="multiple",
                file_types=["image"],
            )
            technique = gr.Radio(
                [TECH_CLOAKING, TECH_CONCEPT, TECH_UNLEARNABLE],
                label="Protection technique",
                value=TECH_CLOAKING,
            )
            epsilon = gr.Slider(minimum=0.01, maximum=0.1, step=0.01, value=0.05, label="Epsilon")
            target_concept = gr.Textbox(label="Target concept for Concept Poisoning", value="a photo of a cat")
            pgd_steps = gr.Slider(minimum=10, maximum=20, step=1, value=12, label="PGD steps")
            processing_mode = gr.Radio(["resize", "patch"], label="Processing mode", value="resize")
            noise_scale = gr.Slider(minimum=5, maximum=30, step=5, value=10, label="Noise visualization scale")
            run_btn = gr.Button("Protect image(s)", variant="primary")
        with gr.Column(scale=2):
            metrics = gr.Textbox(label="Metrics and notes", lines=12)

    with gr.Row():
        original_gallery = gr.Gallery(label="Original", columns=3, height=260)
        protected_gallery = gr.Gallery(label="Protected", columns=3, height=260)
        noise_gallery = gr.Gallery(label="Noise map", columns=3, height=260)

    run_btn.click(
        fn=protect_images,
        inputs=[input_files, technique, epsilon, target_concept, pgd_steps, processing_mode, noise_scale],
        outputs=[original_gallery, protected_gallery, noise_gallery, metrics],
    )

demo.queue(concurrency_count=1).launch(share=True)
